# 4.2 模型参数的访问、初始化和共享

In [29]:
import torch
from torch import nn
from torch.nn import init

print(torch.__version__)

1.13.1+cpu


In [30]:
net = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 1))  # pytorch已进行默认初始化

print(net)
X = torch.rand(2, 4)
Y = net(X).sum()

Sequential(
  (0): Linear(in_features=4, out_features=3, bias=True)
  (1): ReLU()
  (2): Linear(in_features=3, out_features=1, bias=True)
)


## 4.2.1 访问模型参数

In [31]:
print(type(net.named_parameters()))
# 【out_features,in_features】
for name, param in net.named_parameters():
    print(name, param.size())

<class 'generator'>
0.weight torch.Size([3, 4])
0.bias torch.Size([3])
2.weight torch.Size([1, 3])
2.bias torch.Size([1])


In [32]:
for name, param in net[0].named_parameters():
    print(name, param.size(), type(param))

weight torch.Size([3, 4]) <class 'torch.nn.parameter.Parameter'>
bias torch.Size([3]) <class 'torch.nn.parameter.Parameter'>


In [33]:
class MyModel(nn.Module):
    def __init__(self, **kwargs):
        super(MyModel, self).__init__(**kwargs)
        self.weight1 = nn.Parameter(torch.rand(20, 20))
        self.weight2 = torch.rand(20, 20)
    def forward(self, x):
        pass
    
n = MyModel()
for name, param in n.named_parameters():
    print(name)

weight1


In [34]:
weight_0 = list(net[0].parameters())[0] # weight不包含bias

print(list(net[0].parameters()))
print(weight_0.data)
print(weight_0.grad)
Y.backward()
print(weight_0.grad)

[Parameter containing:
tensor([[ 0.1950, -0.2131,  0.4866, -0.1637],
        [ 0.0082,  0.4882, -0.3028,  0.3636],
        [ 0.1357,  0.0524,  0.0141,  0.3892]], requires_grad=True), Parameter containing:
tensor([0.3312, 0.1661, 0.2702], requires_grad=True)]
tensor([[ 0.1950, -0.2131,  0.4866, -0.1637],
        [ 0.0082,  0.4882, -0.3028,  0.3636],
        [ 0.1357,  0.0524,  0.0141,  0.3892]])
None
tensor([[-0.1943, -0.3566, -0.4004, -0.1072],
        [-0.3628, -0.6660, -0.7477, -0.2001],
        [ 0.1082,  0.1986,  0.2230,  0.0597]])


## 4.2.2 初始化模型参数

In [ ]:
for name, param in net.named_parameters():
    if 'weight' in name:
        init.normal_(param, mean=0, std=0.01)
        print(name, param.data)

0.weight torch.Size([3, 4])
0.weight tensor([[-0.0076,  0.0352, -0.0045, -0.0052],
        [-0.0099,  0.0036,  0.0014,  0.0106],
        [ 0.0106, -0.0149,  0.0089, -0.0021]])
0.bias torch.Size([3])
2.weight torch.Size([1, 3])
2.weight tensor([[-0.0036,  0.0048, -0.0049]])
2.bias torch.Size([1])


In [36]:
for name, param in net.named_parameters():
    if 'bias' in name:
        init.constant_(param, val=0)
        print(name, param.data)

0.bias tensor([0., 0., 0.])
2.bias tensor([0.])


## 4.2.3 自定义初始化方法

In [37]:
def init_weight_(tensor):
    with torch.no_grad():
        tensor.uniform_(-10, 10)
        tensor *= (tensor.abs() >= 5).float()

In [38]:
for name, param in net.named_parameters():
    if 'weight' in name:
        init_weight_(param)
        print(name, param.data)

0.weight tensor([[ 0.0000, -0.0000,  0.0000,  9.4184],
        [ 8.3305, -0.0000, -0.0000, -7.4002],
        [-8.9187, -0.0000,  0.0000, -0.0000]])
2.weight tensor([[-8.2620,  0.0000, -9.1452]])


In [39]:
for name, param in net.named_parameters():
    if 'bias' in name:
        param.data += 1
        print(name, param.data)

0.bias tensor([1., 1., 1.])
2.bias tensor([1.])


## 4.2.4 共享模型参数

In [40]:
linear = nn.Linear(1, 1, bias=False)
net = nn.Sequential(linear, linear) 
print(net)
for name, param in net.named_parameters():
    init.constant_(param, val=3)
    print(name, param.data)

Sequential(
  (0): Linear(in_features=1, out_features=1, bias=False)
  (1): Linear(in_features=1, out_features=1, bias=False)
)
0.weight tensor([[3.]])


In [41]:
print(id(net[0]) == id(net[1]))
print(id(net[0].weight) == id(net[1].weight))

True
True


In [42]:
x = torch.ones(1, 1)
y = net(x).sum()
print(y)
y.backward()
print(net[0].weight.grad)

tensor(9., grad_fn=<SumBackward0>)
tensor([[6.]])
